In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
def update_parameters_with_gd(parameters, grads, learning_rate):
    if learning_rate <= 0:
        raise ValueError('learning_rate는 양수여야 합니다.')
    L = len(parameters) // 2
    for l in range(1, L + 1):
        assert parameters['W' + str(l)].shape == grads['dW' + str(l)].shape
        assert parameters['b' + str(l)].shape == grads['db' + str(l)].shape
        parameters['W' + str(l)] = parameters['W' + str(l)] - learning_rate * grads['dW' + str(l)]
        parameters['b' + str(l)] = parameters['b' + str(l)] - learning_rate * grads['db' + str(l)]
    return parameters


In [ ]:
def get_batches(X, y, batch_size, shuffle=True, rng=None):
    assert X.ndim == 2 and y.shape == (len(X),)
    if batch_size < 1 or not isinstance(batch_size, (int, np.integer)):
        raise ValueError('batch_size는 양의 정수여야 합니다.')
    if rng is None:
        rng = np.random.default_rng()
    indices = rng.permutation(len(X)) if shuffle else np.arange(len(X))
    for start in range(0, len(X), batch_size):
        selected = indices[start:start + batch_size]
        yield (X[selected], y[selected])

def random_mini_batches(X, Y, mini_batch_size=64, seed=0):
    return list(get_batches(X, Y, mini_batch_size, rng=np.random.default_rng(seed)))


In [ ]:
def load_data(n_samples=1200, noise=0.12, seed=3):
    if n_samples < 20 or noise < 0:
        raise ValueError('n_samples >= 20, noise >= 0 이어야 합니다.')
    rng = np.random.default_rng(seed)
    y = np.arange(n_samples) % 2
    angles = rng.uniform(0.0, 2.0 * np.pi, size=n_samples)
    radii = np.where(y == 0, 1.0, 0.45)
    X = np.column_stack((radii * np.cos(angles), radii * np.sin(angles)))
    X += rng.normal(0.0, noise, size=X.shape)
    order = rng.permutation(n_samples)
    return (X[order].astype(np.float64), y[order].astype(np.int64))

def split_data(X, y, test_size=0.2, seed=10):
    assert X.ndim == 2 and y.shape == (len(X),)
    if not 0.0 < test_size < 1.0:
        raise ValueError('test_size는 0과 1 사이여야 합니다.')
    rng = np.random.default_rng(seed)
    train_indices, test_indices = ([], [])
    for label in np.unique(y):
        indices = rng.permutation(np.flatnonzero(y == label))
        n_test = int(round(len(indices) * test_size))
        if n_test == 0 or n_test == len(indices):
            raise ValueError('각 클래스에 train/test 샘플이 필요합니다.')
        test_indices.extend(indices[:n_test])
        train_indices.extend(indices[n_test:])
    train_indices = rng.permutation(train_indices)
    test_indices = rng.permutation(test_indices)
    return (X[train_indices], X[test_indices], y[train_indices], y[test_indices])

def standardize(X_train, *other_sets):
    mean = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True)
    std = np.where(std < 1e-12, 1.0, std)
    scaled = tuple(((X - mean) / std for X in (X_train,) + other_sets))
    return (scaled, {'mean': mean, 'std': std})

def load_dataset(n_samples=1200, noise=0.12, seed=3):
    X, y = load_data(n_samples, noise, seed)
    X_pool, X_test, y_pool, y_test = split_data(X, y, 0.2, seed=10)
    X_train, X_val, y_train, y_val = split_data(X_pool, y_pool, 0.2, seed=11)
    (X_train, X_val, X_test), stats = standardize(X_train, X_val, X_test)
    return (X_train, y_train, X_val, y_val, X_test, y_test, stats)


In [ ]:
class DenseLayer:

    def __init__(self, in_features, out_features, initialization='he', rng=None):
        rng = np.random.default_rng(3) if rng is None else rng
        if initialization == 'he':
            scale = np.sqrt(2.0 / in_features)
        elif initialization == 'xavier':
            scale = np.sqrt(1.0 / in_features)
        else:
            raise ValueError('initialization은 he 또는 xavier입니다.')
        self.W = rng.normal(size=(in_features, out_features)) * scale
        self.b = np.zeros((1, out_features), dtype=np.float64)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self.initialization = initialization
        self.cache = None

    def forward(self, X):
        assert X.ndim == 2 and X.shape[1] == self.W.shape[0]
        out = X @ self.W + self.b
        self.cache = {'X': X, 'out_shape': out.shape}
        assert out.shape == (X.shape[0], self.W.shape[1])
        return out

    def backward(self, dout):
        if self.cache is None:
            raise RuntimeError('forward를 먼저 실행하세요.')
        X = self.cache['X']
        assert dout.shape == self.cache['out_shape']
        self.dW = X.T @ dout
        self.db = np.sum(dout, axis=0, keepdims=True)
        dX = dout @ self.W.T
        assert self.dW.shape == self.W.shape and self.db.shape == self.b.shape
        assert dX.shape == X.shape
        return dX

class ReLU:

    def __init__(self):
        self.cache = None

    def forward(self, X):
        assert X.ndim == 2
        self.cache = X
        out = np.maximum(0.0, X)
        assert out.shape == X.shape
        return out

    def backward(self, dout):
        if self.cache is None:
            raise RuntimeError('forward를 먼저 실행하세요.')
        assert dout.shape == self.cache.shape
        dX = dout * (self.cache > 0)
        assert dX.shape == self.cache.shape
        return dX

class Tanh:

    def __init__(self):
        self.cache = None

    def forward(self, X):
        assert X.ndim == 2
        out = np.tanh(X)
        self.cache = out
        assert out.shape == X.shape
        return out

    def backward(self, dout):
        if self.cache is None:
            raise RuntimeError('forward를 먼저 실행하세요.')
        assert dout.shape == self.cache.shape
        dX = dout * (1.0 - self.cache ** 2)
        assert dX.shape == self.cache.shape
        return dX

class SoftmaxCrossEntropy:

    def __init__(self):
        self.cache = None

    def forward(self, logits, y):
        assert logits.ndim == 2 and logits.shape[0] > 0
        assert y.shape == (len(logits),) and np.issubdtype(y.dtype, np.integer)
        assert np.all((0 <= y) & (y < logits.shape[1]))
        shifted = logits - np.max(logits, axis=1, keepdims=True)
        log_probs = shifted - np.log(np.sum(np.exp(shifted), axis=1, keepdims=True))
        probs = np.exp(log_probs)
        loss = -np.mean(log_probs[np.arange(len(y)), y])
        self.cache = {'probs': probs, 'y': y.copy()}
        assert np.isfinite(loss) and probs.shape == logits.shape
        return float(loss)

    def backward(self):
        if self.cache is None:
            raise RuntimeError('forward를 먼저 실행하세요.')
        probs, y = (self.cache['probs'], self.cache['y'])
        dlogits = probs.copy()
        dlogits[np.arange(len(y)), y] -= 1.0
        dlogits /= len(y)
        assert dlogits.shape == probs.shape
        return dlogits


In [ ]:
class DeepNeuralNetwork:

    def __init__(self, layers):
        self.layers = list(layers)
        if sum((isinstance(layer, DenseLayer) for layer in self.layers)) < 3:
            raise ValueError('최소 3개 Dense(은닉층 2개 + 출력층)가 필요합니다.')
        assert isinstance(self.layers[-1], DenseLayer)

    @property
    def dense_layers(self):
        return [layer for layer in self.layers if isinstance(layer, DenseLayer)]

    def forward(self, X):
        assert X.ndim == 2
        out = X
        for layer in self.layers:
            out = layer.forward(out)
            assert out.ndim == 2 and out.shape[0] == X.shape[0]
        return out

    def backward(self, dout):
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def step(self, learning_rate):
        parameters, grads = ({}, {})
        for l, layer in enumerate(self.dense_layers, 1):
            parameters['W' + str(l)], parameters['b' + str(l)] = (layer.W, layer.b)
            grads['dW' + str(l)], grads['db' + str(l)] = (layer.dW, layer.db)
        parameters = update_parameters_with_gd(parameters, grads, learning_rate)
        for l, layer in enumerate(self.dense_layers, 1):
            layer.W, layer.b = (parameters['W' + str(l)], parameters['b' + str(l)])

def initialize_parameters(layers_dims=(2, 16, 8, 2), seed=3):
    if len(layers_dims) != 4:
        raise ValueError('이번 실험은 입력/은닉1/은닉2/출력의 4개 차원을 받습니다.')
    rng = np.random.default_rng(seed)
    return DeepNeuralNetwork([DenseLayer(layers_dims[0], layers_dims[1], 'he', rng), ReLU(), DenseLayer(layers_dims[1], layers_dims[2], 'xavier', rng), Tanh(), DenseLayer(layers_dims[2], layers_dims[3], 'xavier', rng)])

def forward_propagation(X, network):
    return network.forward(X)

def backward_propagation(dlogits, network):
    return network.backward(dlogits)

def compute_cost(logits, y):
    return SoftmaxCrossEntropy().forward(logits, y)

def predict(network, X):
    return np.argmax(network.forward(X), axis=1)

def evaluate(network, X, y):
    logits = network.forward(X)
    return {'loss': compute_cost(logits, y), 'accuracy': float(np.mean(np.argmax(logits, axis=1) == y))}


In [ ]:
def gradient_check(network, X, y, epsilon=1e-05):
    criterion = SoftmaxCrossEntropy()
    criterion.forward(network.forward(X), y)
    analytic_dX = network.backward(criterion.backward()).copy()
    relu_margin = min((float(np.min(np.abs(layer.cache))) for layer in network.layers if isinstance(layer, ReLU)))
    if relu_margin < 100 * epsilon:
        raise ValueError('ReLU 입력이 0에 가깝습니다. 다른 검사 배치를 사용하세요.')
    analytic = [(layer.dW.copy(), layer.db.copy()) for layer in network.dense_layers]
    rows = []

    def compare(name, parameter, analytic_grad):
        numeric_grad = np.zeros_like(parameter)
        for index in np.ndindex(parameter.shape):
            old_value = parameter[index]
            try:
                parameter[index] = old_value + epsilon
                loss_plus = criterion.forward(network.forward(X), y)
                parameter[index] = old_value - epsilon
                loss_minus = criterion.forward(network.forward(X), y)
                numeric_grad[index] = (loss_plus - loss_minus) / (2.0 * epsilon)
            finally:
                parameter[index] = old_value
        relative_error = np.linalg.norm(analytic_grad - numeric_grad) / (np.linalg.norm(analytic_grad) + np.linalg.norm(numeric_grad) + 1e-12)
        rows.append({'parameter': name, 'shape': list(parameter.shape), 'n_checked': int(parameter.size), 'relative_error': float(relative_error), 'max_absolute_error': float(np.max(np.abs(analytic_grad - numeric_grad)))})
    for l, (layer, (dW, db)) in enumerate(zip(network.dense_layers, analytic), 1):
        compare('W' + str(l), layer.W, dW)
        compare('b' + str(l), layer.b, db)
    compare('X (dX)', X, analytic_dX)
    criterion.forward(network.forward(X), y)
    network.backward(criterion.backward())
    return {'epsilon': epsilon, 'threshold': 1e-06, 'relu_min_abs_input': relu_margin, 'results': rows, 'passed': bool(all((row['relative_error'] < 1e-06 for row in rows)))}


In [ ]:
def model(X, Y, X_val, Y_val, layers_dims=(2, 16, 8, 2), optimizer='gd', learning_rate=0.1, mini_batch_size=32, num_epochs=300, print_cost=True, seed=3):
    if optimizer not in ('gd', 'sgd'):
        raise ValueError('이번 제출물은 필수 optimizer인 SGD를 구현합니다.')
    if num_epochs < 1:
        raise ValueError('num_epochs는 양수여야 합니다.')
    network = initialize_parameters(layers_dims, seed)
    criterion = SoftmaxCrossEntropy()
    history = []
    rng = np.random.default_rng(10)
    for i in range(num_epochs):
        minibatches = get_batches(X, Y, mini_batch_size, shuffle=True, rng=rng)
        cost_total, n_seen = (0.0, 0)
        for minibatch_X, minibatch_Y in minibatches:
            logits = forward_propagation(minibatch_X, network)
            cost = criterion.forward(logits, minibatch_Y)
            backward_propagation(criterion.backward(), network)
            network.step(learning_rate)
            cost_total += cost * len(minibatch_X)
            n_seen += len(minibatch_X)
        assert n_seen == len(X)
        train_metrics = evaluate(network, X, Y)
        val_metrics = evaluate(network, X_val, Y_val)
        row = {'epoch': i + 1, 'batch_loss': cost_total / n_seen, 'train_loss': train_metrics['loss'], 'train_accuracy': train_metrics['accuracy'], 'val_loss': val_metrics['loss'], 'val_accuracy': val_metrics['accuracy']}
        history.append(row)
        if print_cost and (i == 0 or (i + 1) % 50 == 0 or i + 1 == num_epochs):
            print('Epoch %3d | train loss %.4f acc %.2f%% | val loss %.4f acc %.2f%%' % (i + 1, row['train_loss'], 100 * row['train_accuracy'], row['val_loss'], 100 * row['val_accuracy']))
    return (network, history)


In [ ]:
def run_experiments(data, learning_rates=(0.01, 0.1), num_epochs=300, print_cost=True):
    X_train, y_train, X_val, y_val, X_test, y_test, stats = data
    networks, histories, summaries = ({}, {}, [])
    for lr in learning_rates:
        if print_cost:
            print('\nMini-batch SGD, learning_rate =', lr)
        network, history = model(X_train, y_train, X_val, y_val, learning_rate=lr, num_epochs=num_epochs, print_cost=print_cost)
        key = 'lr_' + str(lr)
        networks[key], histories[key] = (network, history)
        summaries.append({'experiment': key, 'learning_rate': lr, 'initial_train_loss': evaluate(initialize_parameters(), X_train, y_train)['loss'], 'train_loss': history[-1]['train_loss'], 'train_accuracy': history[-1]['train_accuracy'], 'val_loss': history[-1]['val_loss'], 'val_accuracy': history[-1]['val_accuracy']})
    best = min(summaries, key=lambda row: row['val_loss'])
    final_test = evaluate(networks[best['experiment']], X_test, y_test)
    return (networks, histories, {'experiments': summaries, 'selected_experiment': best['experiment'], 'selection_rule': 'lowest final validation loss', 'test_loss': final_test['loss'], 'test_accuracy': final_test['accuracy'], 'split_sizes': {'train': len(X_train), 'validation': len(X_val), 'test': len(X_test)}, 'layers_dims': [2, 16, 8, 2], 'epochs': num_epochs, 'batch_size': 32, 'data_seed': 3, 'model_seed': 3, 'batch_seed': 10, 'noise': 0.12, 'standardization': {k: v.tolist() for k, v in stats.items()}})

def plot_learning_curves(histories):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for key, history in histories.items():
        epochs = [row['epoch'] for row in history]
        line, = axes[0].plot(epochs, [row['train_loss'] for row in history], label=key + ' train')
        axes[0].plot(epochs, [row['val_loss'] for row in history], '--', color=line.get_color(), label=key + ' val')
        axes[1].plot(epochs, [row['train_accuracy'] for row in history], color=line.get_color(), label=key + ' train')
        axes[1].plot(epochs, [row['val_accuracy'] for row in history], '--', color=line.get_color(), label=key + ' val')
    for ax, ylabel in zip(axes, ('Softmax cross entropy', 'Accuracy')):
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    axes[0].set_title('Training / validation loss')
    axes[1].set_title('Training / validation accuracy')
    fig.tight_layout()
    return fig

def plot_decision_boundary(network, X, y):
    x_min, x_max = (X[:, 0].min() - 0.4, X[:, 0].max() + 0.4)
    y_min, y_max = (X[:, 1].min() - 0.4, X[:, 1].max() + 0.4)
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 220), np.linspace(y_min, y_max, 220))
    Z = predict(network, np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5], cmap='coolwarm', alpha=0.25)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=20, edgecolors='k', linewidths=0.3)
    ax.set_xlabel('Standardized x1')
    ax.set_ylabel('Standardized x2')
    ax.set_title('NumPy DNN decision boundary (test set)')
    ax.set_aspect('equal')
    fig.tight_layout()
    return fig


In [ ]:
data = load_dataset()
X_train, y_train, X_val, y_val, X_test, y_test, stats = data
for name, X, y in (('train', X_train, y_train), ('validation', X_val, y_val), ('test', X_test, y_test)):
    assert X.ndim == 2 and X.shape[1] == 2 and (y.shape == (len(X),))
    print('%10s: X%s, y%s, class counts=%s' % (name, X.shape, y.shape, np.bincount(y, minlength=2)))
print('train mean after standardization:', X_train.mean(axis=0))
print('train std after standardization:', X_train.std(axis=0))
assert np.allclose(X_train.mean(axis=0), 0.0, atol=1e-12)
assert np.allclose(X_train.std(axis=0), 1.0, atol=1e-12)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', s=14, alpha=0.7)
ax.set_title('Train set: NumPy-generated circles')
ax.set_xlabel('Standardized x1')
ax.set_ylabel('Standardized x2')
ax.set_aspect('equal')
plt.show()


In [ ]:
demo_batches = random_mini_batches(X_train, y_train, mini_batch_size=31, seed=10)
assert sum((len(X_batch) for X_batch, _ in demo_batches)) == len(X_train)
assert all((X_batch.ndim == 2 and y_batch.shape == (len(X_batch),) for X_batch, y_batch in demo_batches))
print('Mini-batches: count=%d, first=%s, last=%s' % (len(demo_batches), demo_batches[0][0].shape, demo_batches[-1][0].shape))
demo_network = initialize_parameters()
demo_X, demo_y = (X_train[:5].copy(), y_train[:5].copy())
demo_out = demo_X
print('\nForward shapes and cache:')
for index, layer in enumerate(demo_network.layers, 1):
    demo_out = layer.forward(demo_out)
    assert layer.cache is not None
    print('%d. %-10s -> %s' % (index, type(layer).__name__, demo_out.shape))
demo_criterion = SoftmaxCrossEntropy()
demo_loss = demo_criterion.forward(demo_out, demo_y)
demo_dlogits = demo_criterion.backward()
demo_dX = demo_network.backward(demo_dlogits)
print('loss=%.6f, dlogits=%s, dX=%s' % (demo_loss, demo_dlogits.shape, demo_dX.shape))
assert demo_dX.shape == demo_X.shape
print('\nDense parameter / gradient shapes:')
for index, layer in enumerate(demo_network.dense_layers, 1):
    print('Dense %d: W=%s dW=%s | b=%s db=%s' % (index, layer.W.shape, layer.dW.shape, layer.b.shape, layer.db.shape))
    assert layer.W.shape == layer.dW.shape and layer.b.shape == layer.db.shape
demo_lr = 0.1
expected_parameters = [(layer.W - demo_lr * layer.dW, layer.b - demo_lr * layer.db) for layer in demo_network.dense_layers]
demo_network.step(demo_lr)
for layer, (expected_W, expected_b) in zip(demo_network.dense_layers, expected_parameters):
    assert np.allclose(layer.W, expected_W) and np.allclose(layer.b, expected_b)
print('SGD verification: passed (parameter - learning_rate * gradient)')


In [ ]:
gradient_result = gradient_check(initialize_parameters(), X_train[:5].copy(), y_train[:5])
print('epsilon=%.1e, threshold=%.1e, minimum |ReLU input|=%.6e' % (gradient_result['epsilon'], gradient_result['threshold'], gradient_result['relu_min_abs_input']))
print('\n%-12s %-10s %9s %18s %18s' % ('parameter', 'shape', 'checked', 'relative error', 'max abs error'))
for row in gradient_result['results']:
    print('%-12s %-10s %9d %18.10e %18.10e' % (row['parameter'], str(tuple(row['shape'])), row['n_checked'], row['relative_error'], row['max_absolute_error']))
print('Gradient check passed:', gradient_result['passed'])
assert gradient_result['passed'], '수치미분과 역전파 gradient가 일치하지 않습니다.'


In [ ]:
networks, histories, summary = run_experiments(data)
print('\nFinal experiment results:')
print('%-10s %10s %12s %12s %12s %12s' % ('experiment', 'initial L', 'train loss', 'train acc', 'val loss', 'val acc'))
for row in summary['experiments']:
    print('%-10s %10.6f %12.6f %11.2f%% %12.6f %11.2f%%' % (row['experiment'], row['initial_train_loss'], row['train_loss'], 100 * row['train_accuracy'], row['val_loss'], 100 * row['val_accuracy']))
print('Selected by final validation loss:', summary['selected_experiment'])
print('Final test: loss=%.6f, accuracy=%.2f%%' % (summary['test_loss'], 100 * summary['test_accuracy']))
assert all((len(history) == 300 for history in histories.values()))


In [ ]:
learning_figure = plot_learning_curves(histories)
plt.show()
selected_network = networks[summary['selected_experiment']]
boundary_figure = plot_decision_boundary(selected_network, X_test, y_test)
plt.show()
